# Archived-result reproduction
Set `RAFTUP_EXPERIMENT_ROOT` to the revision data archive root before running. This notebook redraws saved results, not model training. Outputs are separate from the archive. See ../README.md.


# Final-alignment spatial distortion: prevalence and excess

从已有结果重绘；不训练、不重新配准。先运行公共路径、数据读取与绘图设置，再运行独立绘图 cell。

输入根目录在第一个代码 cell 的 `ROOT`；所有新结果写入本文件夹的 `outputs/`，不覆盖原始结果。主要绘图代码已复制进本 notebook，可直接修改。

原始文件：
- `/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/distortion_violation_audit/final_alignment_cutoff_violations.ipynb`

论文对应图片：`violation.png`。

本文件仅有一张堆叠条形图，五个数据集保留在同一坐标轴中。放大字体、不添加字母标注，零外围留白导出到 `outputs/05_spatial_distortion/panels/`，包含 PDF、SVG、600 dpi PNG 和预览图。


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/raftup_reviewer_figures_mpl')
# Input project root. Change only this path when moving the project.
ROOT = Path(os.environ.get('RAFTUP_EXPERIMENT_ROOT', './revision_data')).expanduser().resolve()
NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', './outputs')).expanduser().resolve()
OUT = RESULTS_DIR / '05_spatial_distortion'
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Missing input project: {ROOT}'
print('Input project:', ROOT)
print('Outputs:', OUT)



## Combined violation prevalence and excess magnitude

One stacked bar per dataset combines the final excess-distribution plots. Each segment is the fraction of **all distinct positive-support correspondence-edge pairs** falling in an excess interval, averaged with equal weight per experiment. Bar lengths therefore reproduce the ordinary full-cutoff violation rates; these are not conditional percentages among violations. Zero-violation experiments contribute zero to every segment.

Excess is |dA − dB| minus the experiment’s full cutoff. Intervals align exactly with the saved 1-μm histogram bins. Full recovery thresholds are numerical pairwise references, not tests of anchor-residual feasibility. Deep-STARmap uses the saved historical μm convention. This cell runs independently from saved CSVs without rerunning alignment or pairwise enumeration.


In [ ]:
# Combined prevalence and excess severity; saved tables only, no audit rerun.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
base = ROOT / 'all_exp/distortion_violation_audit/results_all'
order = ['DLPFC', 'MERFISH', 'Simulated overlap', 'Cropped windows', 'Deep-STARmap']
bounds = np.array([0., 10., 25., 50., 100., np.inf])
bin_labels = ['0–<10', '10–<25', '25–<50', '50–<100', '≥100']
colors = ['#c6dbef', '#6baed6', '#3182bd', '#e89442', '#b74432']
dlpfc_dir = base / 'dlpfc_distance_diagnostics'
dlpfc = pd.read_csv(dlpfc_dir / 'dlpfc_violation_distance_summary.csv', dtype={'pair': str}).rename(columns={'pair': 'case'})
dlpfc['dataset'] = 'DLPFC'
other_dir = base / 'other_dataset_excess_diagnostics'
other = pd.read_csv(other_dir / 'per_case_excess_summary.csv', dtype={'case': str})
stats = pd.concat([dlpfc, other], ignore_index=True)
assert len(stats) == 51 and not stats.duplicated(['dataset', 'case']).any()
rows = []
for r in stats.itertuples():
    stem = r.dataset.lower().replace(' ', '_').replace('-', '_')
    folder = dlpfc_dir if r.dataset == 'DLPFC' else other_dir / stem
    hist = pd.read_csv(folder / f'{r.case}_excess_histogram.csv')
    left = hist.excess_bin_left_um.to_numpy()
    right = hist.excess_bin_right_um.to_numpy()
    count = hist['count'].to_numpy()
    assert r.total_combinations > 0 and count.sum() == r.violating_combinations
    np.testing.assert_allclose(right - left, 1.)
    # New boundaries coincide with saved 1-μm bin edges: no interpolation.
    segments = []
    for low, high, label in zip(bounds[:-1], bounds[1:], bin_labels):
        mask = (left >= low) & (left < high)
        assert np.all(right[mask] <= high)
        fraction = count[mask].sum() / r.total_combinations
        segments.append(fraction)
        rows.append(dict(dataset=r.dataset, case=r.case, excess_bin_um=label,
                         fraction_all_edge_pairs=fraction, percent_all_edge_pairs=100*fraction))
    np.testing.assert_allclose(sum(segments), r.ordinary_violation_rate, atol=1e-12)
per_case = pd.DataFrame(rows)
combined = per_case.groupby(['dataset', 'excess_bin_um']).percent_all_edge_pairs.mean().unstack().reindex(index=order, columns=bin_labels)
expected = pd.read_csv(base / 'dataset_mean_ordinary_violation.csv').set_index('dataset').reindex(order)
np.testing.assert_allclose(combined.sum(axis=1), expected.mean_violation_percent, atol=1e-10)
counts = stats.groupby('dataset').size().reindex(order)
per_case.to_csv(OUT / 'combined_violation_excess_per_case.csv', index=False)
table = combined.copy()
table['total_violation_percent'] = combined.sum(axis=1)
table['n_experiments'] = counts
table.to_csv(OUT / 'combined_violation_excess_summary.csv')

display(table)


In [ ]:
# Shared plot settings. This notebook contains one stacked-bar panel.
FIG = OUT / 'panels'
FIG.mkdir(exist_ok=True)
plt.rcParams.update({
    'font.family': 'Arial', 'font.size': 17,
    'axes.labelsize': 18, 'axes.titlesize': 20,
    'xtick.labelsize': 16, 'ytick.labelsize': 17,
    'legend.fontsize': 15, 'legend.title_fontsize': 17,
    'axes.spines.top': False, 'axes.spines.right': False,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.facecolor': 'white',
})

def save_panel(fig, name):
    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(FIG / f'{name}.{ext}', dpi=600,
                    bbox_inches='tight', pad_inches=0)
    preview = FIG / f'{name}_preview.png'
    fig.savefig(preview, dpi=180, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    display(Image(filename=str(preview)))
    print(FIG / f'{name}.pdf')


## Spatial-cutoff violations — 独立绘图 cell


In [ ]:
# Stacked bars: total violation prevalence and excess magnitude.
fig, ax = plt.subplots(figsize=(13, 6.8))
fig.subplots_adjust(left=.29, right=.97, top=.87, bottom=.31)
y = np.arange(len(order))
cumulative = np.zeros(len(order))
for label, color in zip(bin_labels, colors):
    values = combined[label].to_numpy()
    ax.barh(y, values, left=cumulative, height=.58, color=color,
            edgecolor='white', linewidth=.7, label=label)
    cumulative += values
for i, total in enumerate(cumulative):
    ax.text(total + .14, i, f'{total:.2f}%', va='center',
            fontsize=17, fontweight='bold')
    if total == 0:
        ax.plot(0, i, '|', color='#64748b', markersize=16, clip_on=False)
labels = [f'{name}  ({counts[name]} pairs)' for name in order]
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlim(0, max(1., cumulative.max()) * 1.18)
ax.set_xlabel('Mean fraction of all correspondence-edge pairs (%)', labelpad=12)
ax.grid(axis='x', alpha=.2)
ax.set_axisbelow(True)
ax.tick_params(axis='y', length=0, pad=10)
ax.legend(title='Excess above predefined cutoff (μm)', ncol=5,
          loc='upper center', bbox_to_anchor=(.5, -.28),
          frameon=False, borderaxespad=0, columnspacing=1.1, handlelength=1.4)
ax.set_title('Spatial-cutoff violations across experiments',
             fontsize=20, fontweight='bold', pad=20)
save_panel(fig, 'combined_violation_excess')
